In [1]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"

import torch
import gc
import json
import numpy as np
from tqdm import tqdm
import evaluate
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

print("CUDA Available:", torch.cuda.is_available())

Skipping import of cpp extensions due to incompatible torch version 2.6.0+cu124 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
C:\Users\Legion\AppData\Local\Temp\ipykernel_25976\4195150846.py:10: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
CUDA Available: True


In [2]:
BASE_MODEL_DIR = "base_llama3_model"
ADAPTER_DIR = "adapters/ENRON_Adapter"
TEST_FILE = "phase1b_enron_test_reasoning_filtered.jsonl" 

In [3]:
print("\nLoading models for Enron inference...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL_DIR, 
    max_seq_length = 2048, # Optimized for email text
    dtype = None,
    load_in_4bit = True,
)

print(f"Loading Enron LoRA Adapter from: {ADAPTER_DIR}...")
model.load_adapter(ADAPTER_DIR)
FastLanguageModel.for_inference(model)


Loading models for Enron inference...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading Enron LoRA Adapter from: adapters/ENRON_Adapter...


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 3072, padding_idx=128004)
    (layers): ModuleList(
      (0-27): 28 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
            (lora_dropout): ModuleDict(
              (default): Identity()
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=3072, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=3072, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): Linear4bit(in_features=3072, out_features=1024, bias=False)
          (v_proj): lora.Linear4bit(
            (base_layer): Linear4bit(in_features=307

In [4]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an AI Executive Assistant. Analyze the following corporate email thread.
Determine the sender's core intent, summarize the context, and extract ALL required action items.

CRITICAL RULES:
1. Identify who needs to do what.
2. If there are multiple action items, list all of them.
3. If the action matches a system function (e.g., schedule_meeting), use it. If it is a custom human task, describe it clearly in plain text.

Format your output EXACTLY like this:
Thought: [Your summary and intent analysis]
Action: [The extracted action item(s), tasks, or functions]

### Input:
{}

### Response:
{}"""

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

In [5]:
print(f"\nLoading test data from {TEST_FILE}...")
test_samples = []
if os.path.exists(TEST_FILE):
    with open(TEST_FILE, "r", encoding="utf-8") as f:
        for line in f:
            test_samples.append(json.loads(line))
else:
    print(f"Warning: {TEST_FILE} not found. Only the custom email test will run.")

pred_strings = []
ref_strings = []


Loading test data from phase1b_enron_test_reasoning_filtered.jsonl...


In [6]:
if test_samples:
    print(f"\nAgent is evaluating {len(test_samples)} unseen corporate emails...")
    for sample in tqdm(test_samples, desc="Evaluating Emails"): 
        unseen_email = sample["input"]
        gold_full_text = sample["output"] 
        
        inputs = tokenizer([alpaca_prompt.format(unseen_email, "")], return_tensors="pt").to("cuda")
        
        outputs = model.generate(
            **inputs, 
            max_new_tokens = 500, # Email are short 500 tokens is enoug   
            temperature = 0.1,          
            top_p = 0.9,                
            use_cache = True,
            repetition_penalty = 1.1,
            eos_token_id = terminators, 
        )
        
        generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
        
        try:
            ai_full_text = generated_text.split("### Response:\n")[1].strip()
        except IndexError:
            ai_full_text = generated_text.strip()
            
        pred_strings.append(ai_full_text)
        ref_strings.append(gold_full_text)

    print("\nLoading BERTScore Model...")
    bertscore = evaluate.load("bertscore")

    # evaluation
    scores = bertscore.compute(predictions=pred_strings, references=ref_strings, lang="en")
    overall_f1 = np.mean(scores['f1']) * 100

    print("\n" + "="*50)
    print("ENRON INTENT & ACTION ACCURACY (BERTScore F1)")
    print("="*50)
    print(f"OVERALL PIPELINE SCORE: {overall_f1:.2f}%")
    print("="*50)


Agent is evaluating 1000 unseen corporate emails...


Evaluating Emails:  16%|█▌        | 160/1000 [13:39<1:11:28,  5.11s/it]Unsloth: Input IDs of shape torch.Size([1, 2222]) with length 2222 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Evaluating Emails:  62%|██████▏   | 623/1000 [51:51<24:56,  3.97s/it]Unsloth: Input IDs of shape torch.Size([1, 2166]) with length 2166 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.
Evaluating Emails: 100%|██████████| 1000/1000 [1:22:35<00:00,  4.96s/it]



Loading BERTScore Model...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



ENRON INTENT & ACTION ACCURACY (BERTScore F1)
OVERALL PIPELINE SCORE: 92.14%


In [7]:
def evaluate_custom_email(email_text, title):
    print(f"\nEvaluating: {title}...\n" + "="*50)
    
    inputs = tokenizer([alpaca_prompt.format(email_text, "")], return_tensors = "pt").to("cuda")
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens = 500,       
        temperature = 0.1,          
        top_p = 0.9,                
        use_cache = True,
        repetition_penalty = 1.1,   
        eos_token_id = terminators, 
    )

    generated_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    try:
        response_only = generated_text.split("### Response:\n")[1].strip()
        print(response_only)
    except IndexError:
        print(generated_text.strip())
    print("="*50)

In [8]:
# Unseen Custom Email
unseen_corporate_email = "Subject: URGENT: Q3 Financial Reports & Friday Sync\n\nHi Team,\n\nI need everyone to focus. Bob, can you please pull the final Q3 revenue numbers and send them to my inbox by EOD tomorrow? We are presenting to the board on Monday.\n\nAlso, Susan, I think we need to align on the upcoming merger integration timeline. Let's schedule a 15-minute sync for this Friday at 2:00 PM to hammer out the details.\n\nThanks,\nRichard"

evaluate_custom_email(unseen_corporate_email, "Urgent Management Request")


Evaluating: Urgent Management Request...
Thought: The sender is requesting urgent action to provide final Q3 revenue numbers by end of day tomorrow and scheduling a 15-minute meeting on Friday at 2:00 PM to discuss the merger integration timeline. The key dates are today (for the report) and this Friday at 2:00 PM for the meeting.

Action: schedule_meeting(time="Friday 2:00 PM", duration="15 minutes")


In [9]:
# HARD SHUTDOWN & UNLOAD FROM GPU

import torch
import gc

print("Initiating VRAM Hard-Shutdown for Local Models...")

# 1. Track memory before cleanup
if torch.cuda.is_available():
    vram_before = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated Before: {vram_before:.2f} GB")

# 2. List of every heavy object that might be trapped in memory
heavy_objects = [
    'model', 'tokenizer', 'trainer', 'bertscore', 'rouge', 
    'dataset', 'train_dataset', 'eval_dataset', 'split_dataset',
    'inputs', 'outputs'
]

# 3. Delete them dynamically if they exist
for obj in heavy_objects:
    if obj in globals():
        print(f"Unloading '{obj}' from memory...")
        del globals()[obj]

# 4. Force aggressive Garbage Collection (Run twice to clear circular references)
gc.collect()
gc.collect()

# 5. Flush the GPU Cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # Clears memory shared between backend processes
    torch.cuda.synchronize()  # Waits for all GPU operations to completely finish
    
    # Track memory after cleanup
    vram_after = torch.cuda.memory_allocated() / 1024**3
    print(f"VRAM Allocated After:  {vram_after:.2f} GB")

print("\nGPU Memory Cleared. Your VRAM is now completely empty!")

Initiating VRAM Hard-Shutdown for Local Models...
VRAM Allocated Before: 3.27 GB
Unloading 'model' from memory...
Unloading 'tokenizer' from memory...
Unloading 'bertscore' from memory...
Unloading 'inputs' from memory...
Unloading 'outputs' from memory...
VRAM Allocated After:  2.27 GB

GPU Memory Cleared. Your VRAM is now completely empty!
